# Module 1 · Lesson 04: Compare Models Side by Side

A key skill for any AI engineer is **choosing the right model** for the job.
In this notebook we send the *same* prompt to multiple models and compare results.

## What you will learn
1. How to call different providers with a **unified interface**
2. Compare **quality, speed, and cost** across models
3. Build a reusable comparison framework
4. Understand the **speed vs quality vs cost** trade-off

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

openai_client = OpenAI()

if openai_client:
  print("OpenAI client is here")

anthropic_client = None
try:
  import anthropic
  if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_client = anthropic.Client()
    print("OpenAI + Anthropic is ready")
  else:
    print("OpenAI is ready | Anthropic is not")  
except ImportError:
      print("Anthropic not installed")
  

OpenAI client is here
OpenAI + Anthropic is ready


---
## 1. Model Registry & Pricing

First, let's define the models and their costs (per 1M tokens, early 2025):

LiteLLM provides a **unified interface** to call any LLM provider with the same code.
Instead of learning separate SDKs, you write one `completion()` call and LiteLLM handles the rest.

```bash
pip install litellm openai anthropic
```

In [ ]:
import os
import json
import litellm
from litellm import completion, completion_cost, model_cost
from IPython.display import display, Markdown

MODELS = [
  "gpt-4o-mini",
  "gpt-4o",
  "gpt-5.2",
  "claude-haiku-4-5",
  "claude-opus-4-6"
]

pricing_md = []

for model in MODELS:
    try:
        info = model_cost[model]
        input_price = info.get("input_cost_per_token") * 1_000_000
        out_price = info.get("output_cost_per_token") * 1_000_000
        pricing_md.append(
          f"- **{model}**: -- Input: '${input_price:.2f}' | Output: '${out_price:.2f}'"
        )  
    except KeyError:
        pricing_md.append(f"**{model}**: Pricing not found") 

display(Markdown("### Model Pricing (per 1 million tokens)\n" + "\n".join(pricing_md)))

### Model Pricing (per 1 million tokens)
- **gpt-4o-mini**: -- Input: '$0.15' | Output: '$0.60'
- **gpt-4o**: -- Input: '$2.50' | Output: '$10.00'
- **gpt-5.2**: -- Input: '$1.75' | Output: '$14.00'
- **claude-haiku-4-5**: -- Input: '$1.00' | Output: '$5.00'
- **claude-opus-4-6**: -- Input: '$5.00' | Output: '$25.00'

---
### Sending the Same Prompt to Every Model

With `litellm.completion()` we can send the exact same prompt to OpenAI and Anthropic models
using **identical code**. LiteLLM translates the call to each provider's native API behind the scenes.

In [11]:
prompt = "Explain what an API is in one paragraph"

for model in MODELS:
    try:
        response = completion(
            model=model, 
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500
        )
        reply = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        cost = completion_cost(completion_response=response)

        md = f"""### {model}
**{reply}**

- Input Tokens: '{input_tokens}' | Output Tokens: '{output_tokens}' | Cost: '${cost:.6f}'
"""  
        display(Markdown(md))
    except Exception as e:
        pass

### gpt-4o-mini
**An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and interact with each other. It defines the methods and data formats that applications can use to request and exchange information, enabling developers to integrate external services, data, or functionality into their own applications without needing to understand the underlying code. APIs can be used for various purposes, such as accessing web services, retrieving data from databases, or controlling hardware devices, making them essential tools for modern software development and collaboration between different systems.**

- Input Tokens: '15' | Output Tokens: '106' | Cost: '$0.000066'


### gpt-4o
**An API, or Application Programming Interface, is a set of predefined rules and protocols that allow different software applications to communicate with each other. It acts as an intermediary layer that enables developers to access certain features or data of a system without needing to understand its internal workings. By exposing a collection of functions and commands, APIs enable the integration and interaction of various applications, allowing them to leverage each other's capabilities efficiently. This is essential in modern software development, facilitating interoperability and the creation of complex systems from modular components distributed across different platforms and services.**

- Input Tokens: '15' | Output Tokens: '107' | Cost: '$0.001107'


### gpt-5.2
**An API (Application Programming Interface) is a defined set of rules and endpoints that lets different software systems communicate with each other in a consistent, controlled way. It specifies what requests a program can make (such as retrieving data, creating a record, or triggering an action), what inputs are allowed, and what responses will be returned, so developers can use a service or component without needing to know its internal implementation.**

- Input Tokens: '14' | Output Tokens: '85' | Cost: '$0.001215'


### claude-haiku-4-5
**# What is an API?

An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate with each other. It acts as a bridge between systems, enabling one program to request data or functionality from another without needing to understand how that other program works internally. For example, when you check the weather on your phone, the app uses an API to request weather data from a weather service's servers, which then sends back the information in a format the app can display. APIs are fundamental to modern software development, allowing developers to build on existing services, integrate third-party tools, and create seamless user experiences across different platforms and devices.**

- Input Tokens: '16' | Output Tokens: '141' | Cost: '$0.000721'


### claude-opus-4-6
**An **API (Application Programming Interface)** is a set of rules, protocols, and tools that allows different software applications to communicate and share data with each other. It acts as an intermediary, defining the methods and data formats that programs can use to request and exchange information, without needing to understand each other's internal workings. For example, when you use a weather app on your phone, it sends a request through an API to a remote server, which then returns the weather data in a structured format that the app can display. APIs are fundamental to modern software development because they enable developers to leverage existing services and functionalities—such as payment processing, mapping, or social media integration—without having to build everything from scratch, promoting efficiency, modularity, and interoperability across systems.**

- Input Tokens: '16' | Output Tokens: '164' | Cost: '$0.004180'


---
### Multi-Turn Conversation with Cost Tracking

LiteLLM also supports multi-turn conversations. We can track the cumulative cost across turns.

In [13]:
model = "gpt-4o-mini"
total_cost = 0.0

conversation = [
    {"role": "user", "content": "Hello. My name is Georgios"},
]

# Turn 1
response_turn1 = completion(
    model=model,
    messages=conversation,
    max_tokens=150
)
assistant_reply1 = response_turn1.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn1) 

display(Markdown(f"""
**User**: Hello. My name is Georgios.
                 
**Assistant:** {assistant_reply1}
"""))


**User**: Hello. My name is Georgios.

**Assistant:** Hello, Georgios! How can I assist you today?


In [16]:
# Turn 2
conversation.append({"role": "assistant", "content": assistant_reply1})
conversation.append({"role": "user", "content": "What is my name?"}) 

response_turn2 = completion(
    model=model,
    messages=conversation,
    max_tokens=150
)
assistant_reply2 = response_turn2.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn2)

display(Markdown(f"""
**User:** What is my name?
                 
**Assistant:** {assistant_reply2}

**Total cost:** '${total_cost:.6f}'
"""
))


**User:** What is my name?

**Assistant:** Your name is Georgios. How can I help you today?

**Total cost:** '$0.000062'


---
### Cost Comparison for the Same Prompt

Let's send the same creative prompt to every model and compare costs side by side, sorted cheapest first.

In [17]:
# A haiku has 3 lines with a specific syllable pattern
# Line 1 → 5 syllables
# Line 2 → 7 syllables
# Line 3 → 5 syllables

prompt = "Write a haiku about programming."
results = []

# Run prompt across models
for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
        )
        cost = completion_cost(completion_response=response)
        reply = response.choices[0].message.content
        results.append({"model": model, "cost": cost, "reply": reply})
    except Exception as e:
        results.append({"model": model, "cost": None, "reply": f"Error: {e}"})

# Sort by cost (cheapest first)
results.sort(key=lambda x: x["cost"] if x["cost"] is not None else float("inf"))

# Display results using Markdown
display(Markdown("## Cost comparison for the same prompt"))

for r in results:
    cost_str = f"${r['cost']:.6f}" if r["cost"] is not None else "N/A"

    display(Markdown(f"""
### {r['model']}
- Cost: **{cost_str}**

**Response**
```text
{r['reply']}
```
"""))

## Cost comparison for the same prompt


### gpt-4o-mini
- Cost: **$0.000015**

**Response**
```text
Lines of code unfold,  
Logic weaves a silent dance,  
Dreams in bits take shape.
```



### claude-haiku-4-5
- Cost: **$0.000134**

**Response**
```text
Code lines multiply,
Logic bends to my will—wait,
Why won't this compile?
```



### gpt-4o
- Cost: **$0.000235**

**Response**
```text
Zeros and ones dance,  
Logic weaves through silent lines—  
Dreams coded to life.
```



### gpt-5.2
- Cost: **$0.000387**

**Response**
```text
Keys clack in soft loops,  
Logic blooms in silent light—  
Bugs fade with dawn’s break.
```



### claude-opus-4-6
- Cost: **$0.000945**

**Response**
```text
# A Programming Haiku

*Code flows line by line*
*Bugs hide in the silent dark*
*Coffee fuels the fix*
```


---
## 2. Unified Call Function

To compare models fairly, we need a function that calls *any* provider and returns standardized results:

In [18]:
# Model registry for direct SDK calls (Sections 2-5)

import time

MODEL_REGISTRY = {
    "gpt-4o-mini":       {"provider": "openai",    "input": 0.15,  "output": 0.60},
    "gpt-4o":            {"provider": "openai",    "input": 2.50,  "output": 10.00},
    "claude-sonnet-4-6": {"provider": "anthropic", "input": 3.00,  "output": 15.00},
    "claude-haiku-4-5":  {"provider": "anthropic", "input": 1.00,  "output": 5.00},
    "claude-opus-4-6":   {"provider": "anthropic", "input": 5.00,  "output": 25.00},
}

def estimate_cost(model, input_tokens, output_tokens):
    info = MODEL_REGISTRY[model]
    return (input_tokens / 1_000_000) * info["input"] + (output_tokens / 1_000_000) * info["output"]

def call_model(prompt: str, model: str, temperature: float = 0.7) -> dict:
    """Call any model and return standardized result dict."""
    info = MODEL_REGISTRY[model]
    start = time.perf_counter()

    if info["provider"] == "openai":
        r = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=temperature
        )
        text = r.choices[0].message.content
        in_tok, out_tok = r.usage.prompt_tokens, r.usage.completion_tokens

    elif info["provider"] == "anthropic":
        r = anthropic_client.messages.create(
            model=model, max_tokens=200, temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        text = r.content[0].text
        in_tok, out_tok = r.usage.input_tokens, r.usage.output_tokens

    elapsed = time.perf_counter() - start
    cost = estimate_cost(model, in_tok, out_tok)

    return {
        "model": model, "provider": info["provider"],
        "text": text, "input_tokens": in_tok, "output_tokens": out_tok,
        "latency_ms": round(elapsed * 1000), "cost_usd": cost
    }

---
## 3. Head-to-Head Comparison

Let's compare all available models on the **same prompt**:

In [19]:
from IPython.display import display, Markdown

# Run comparison
test_prompt = "Explain what a REST API is to a junior developer. Be concise."

display(Markdown(f"##  Model Comparison\n**Prompt:** \"{test_prompt}\"\n---"))

results = []

for model_name in MODEL_REGISTRY:
    try:
        result = call_model(test_prompt, model_name)
        results.append(result)

        # IMPORTANT: do NOT wrap result['text'] in ``` or <pre>
        md = f"""
### 🤖 {model_name} ({result['provider']})

- **Latency:** `{result['latency_ms']} ms`
- **Tokens:** `{result['input_tokens']} in / {result['output_tokens']} out`
- **Cost:** `${result['cost_usd']:.6f}`

---

{result['text']}

---
"""
        display(Markdown(md))

    except Exception as e:
        display(Markdown(f"###  {model_name}\n`Error: {e}`"))

##  Model Comparison
**Prompt:** "Explain what a REST API is to a junior developer. Be concise."
---


### 🤖 gpt-4o-mini (openai)

- **Latency:** `4259 ms`
- **Tokens:** `21 in / 183 out`
- **Cost:** `$0.000113`

---

A REST API (Representational State Transfer Application Programming Interface) is a set of rules that allows different software applications to communicate over the web using standard HTTP methods like GET, POST, PUT, and DELETE. 

In REST:

1. **Resources**: Everything is treated as a resource, identified by a URL (e.g., a user or a product).
2. **Stateless**: Each request from a client to the server must contain all the information needed to understand and process it, meaning the server does not store client context.
3. **Representations**: Resources can be represented in different formats, such as JSON or XML, making it easier for clients to consume the data.
4. **CRUD Operations**: REST APIs typically allow clients to create, read, update, and delete resources.

In summary, a REST API provides a structured way for applications to interact with each other over the web.

---



### 🤖 gpt-4o (openai)

- **Latency:** `1848 ms`
- **Tokens:** `21 in / 122 out`
- **Cost:** `$0.001272`

---

A REST API (Representational State Transfer Application Programming Interface) is a set of rules and conventions for building and interacting with web services. It allows different software applications to communicate with each other over the internet using standard HTTP methods like GET, POST, PUT, and DELETE. REST APIs are stateless, meaning each request from the client contains all the information the server needs to fulfill it, without relying on stored context on the server. They typically return data in formats like JSON or XML, making them easy to use with various programming languages. REST APIs are widely used because they are simple, scalable, and flexible.

---



### 🤖 claude-sonnet-4-6 (anthropic)

- **Latency:** `5882 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.003069`

---

# What is a REST API?

## The Simple Explanation
A REST API is a **standardized way for applications to talk to each other** over the internet using HTTP.

Think of it like a **waiter in a restaurant**:
- You (the client) make a **request**
- The waiter (API) carries it to the kitchen (server)
- The kitchen sends back a **response**

---

## The Key Concepts

**Resources** — Everything is a "thing" identified by a URL
```
/users        → collection of users
/users/42     → a specific user
```

**HTTP Methods** — Define what you want to *do* to that thing
| Method | Action |
|--------|--------|
| `GET` | Read data |
| `POST` | Create something |
| `PUT/PATCH` | Update something |
| `DELETE` | Delete

---



### 🤖 claude-haiku-4-5 (anthropic)

- **Latency:** `3013 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.001023`

---

# REST API Explained

A **REST API** is a set of rules for building web services that allow applications to communicate over HTTP.

## Key Points

**REST** = Representational State Transfer

**Basic idea:** Use standard HTTP methods to perform operations on resources:
- **GET** — retrieve data
- **POST** — create data
- **PUT** — update data
- **DELETE** — remove data

## Simple Example

```
GET /api/users/123        → Get user with ID 123
POST /api/users           → Create a new user
PUT /api/users/123        → Update user 123
DELETE /api/users/123     → Delete user 123
```

## Why It Matters

- **Standardized** — predictable and easy to understand
- **Stateless** — each request is independent
- **Scalable** — works well for distributed systems
-

---



### 🤖 claude-opus-4-6 (anthropic)

- **Latency:** `8298 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.005115`

---

# REST API – A Simple Explanation

A **REST API** is a way for two applications to communicate over the internet using standard **HTTP** methods — the same protocol your browser uses.

## Core Idea
Your app sends a **request** to a URL (called an **endpoint**), and the server sends back a **response** (usually JSON).

## HTTP Methods (CRUD)
| Method | Action | Example |
|--------|---------|---------|
| `GET` | Read data | Fetch a list of users |
| `POST` | Create data | Add a new user |
| `PUT/PATCH` | Update data | Edit a user's name |
| `DELETE` | Remove data | Delete a user |

## Quick Example
```
GET https://api.example.com/users/42
```
→ Returns the user with ID 42:
```json
{ "id":

---


---
## 4. Summary Table

In [20]:
# ── Summary table ─────────────────────────────────────
if results:
    header =  "| Model | Provider | Latency | Tokens | Cost |\n"
    header += "|-------|----------|---------|--------|------|\n"
    rows = ""
    for r in results:
        rows += (f"| {r['model']} | {r['provider']} | {r['latency_ms']} ms | "
                 f"{r['input_tokens']}+{r['output_tokens']} | ${r['cost_usd']:.6f} |\n")
    display(Markdown(header + rows))

    # Cheapest and fastest
    cheapest = min(results, key=lambda x: x['cost_usd'])
    fastest  = min(results, key=lambda x: x['latency_ms'])
    print(f"\n Cheapest: {cheapest['model']} (${cheapest['cost_usd']:.6f})")
    print(f" Fastest:  {fastest['model']} ({fastest['latency_ms']} ms)")

| Model | Provider | Latency | Tokens | Cost |
|-------|----------|---------|--------|------|
| gpt-4o-mini | openai | 4259 ms | 21+183 | $0.000113 |
| gpt-4o | openai | 1848 ms | 21+122 | $0.001272 |
| claude-sonnet-4-6 | anthropic | 5882 ms | 23+200 | $0.003069 |
| claude-haiku-4-5 | anthropic | 3013 ms | 23+200 | $0.001023 |
| claude-opus-4-6 | anthropic | 8298 ms | 23+200 | $0.005115 |



 Cheapest: gpt-4o-mini ($0.000113)
 Fastest:  gpt-4o (1848 ms)


---
## 5. Multiple Test Prompts

A single prompt isn't enough — let's test across different task types:

In [21]:
from IPython.display import display, Markdown

# ── Multi-test comparison ─────────────────────────────
test_suite = [
    ("Factual",    "What is the speed of light in km/s?"),
    ("Creative",   "Write a haiku about programming."),
    ("Analytical", "What are 3 pros and 3 cons of microservices?"),
    ("Reasoning",  "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?"),
]

for test_name, prompt in test_suite:
    display(Markdown(f"""
##  Test: {test_name}
**Prompt:** {prompt}
---
"""))

    for model_name in MODEL_REGISTRY:
        try:
            r = call_model(prompt, model_name, temperature=0.3)

            # Preview (first 200 chars) but keep it markdown-safe and readable
            preview = r["text"][:200] + ("…" if len(r["text"]) > 200 else "")
            preview = preview.replace("\n", " ")  # keep preview on one line

            display(Markdown(f"""
###  {r['model']}
- **Latency:** `{r['latency_ms']} ms`
- **Cost:** `${r['cost_usd']:.6f}`

**Preview:** {preview}

<details>
<summary>Show full response</summary>

{r['text']}

</details>

---
"""))
        except Exception as e:
            display(Markdown(f"###  {model_name}\n`Error: {e}`\n---"))


##  Test: Factual
**Prompt:** What is the speed of light in km/s?
---



###  gpt-4o-mini
- **Latency:** `1661 ms`
- **Cost:** `$0.000024`

**Preview:** The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s). It is often rounded to 300,000 km/s for simplicity in calculations.

<details>
<summary>Show full response</summary>

The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s). It is often rounded to 300,000 km/s for simplicity in calculations.

</details>

---



###  gpt-4o
- **Latency:** `739 ms`
- **Cost:** `$0.000243`

**Preview:** The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s).

<details>
<summary>Show full response</summary>

The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s).

</details>

---



###  claude-sonnet-4-6
- **Latency:** `2332 ms`
- **Cost:** `$0.000549`

**Preview:** The speed of light in a vacuum is approximately **299,792 km/s**, often rounded to **300,000 km/s**.

<details>
<summary>Show full response</summary>

The speed of light in a vacuum is approximately **299,792 km/s**, often rounded to **300,000 km/s**.

</details>

---



###  claude-haiku-4-5
- **Latency:** `1536 ms`
- **Cost:** `$0.000308`

**Preview:** The speed of light is approximately **299,792 km/s** (or about **300,000 km/s** as a rounded figure).  More precisely, it's defined as exactly **299,792.458 km/s** in vacuum.

<details>
<summary>Show full response</summary>

The speed of light is approximately **299,792 km/s** (or about **300,000 km/s** as a rounded figure).

More precisely, it's defined as exactly **299,792.458 km/s** in vacuum.

</details>

---


...


###  gpt-4o-mini
- **Latency:** `3372 ms`
- **Cost:** `$0.000050`

**Preview:** No, we cannot conclude that some roses fade quickly based on the information provided. The statements tell us that all roses are flowers and that some flowers fade quickly, but they do not specify tha…

<details>
<summary>Show full response</summary>

No, we cannot conclude that some roses fade quickly based on the information provided. The statements tell us that all roses are flowers and that some flowers fade quickly, but they do not specify that any of the flowers that fade quickly are roses. Therefore, while it is possible that some roses could fade quickly, it is not a definitive conclusion that can be drawn from the given premises.

</details>

---



###  gpt-4o
- **Latency:** `2448 ms`
- **Cost:** `$0.001538`

**Preview:** The statement "all roses are flowers" establishes that roses are a subset of flowers. The statement "some flowers fade quickly" tells us that within the category of flowers, there is a subset that fad…

<details>
<summary>Show full response</summary>

The statement "all roses are flowers" establishes that roses are a subset of flowers. The statement "some flowers fade quickly" tells us that within the category of flowers, there is a subset that fades quickly. However, it does not specify which types of flowers are in this subset.

To determine if we can conclude that some roses fade quickly, we need to consider the logical relationship between the subsets:

1. All roses are flowers.
2. Some flowers fade quickly.

The second statement does not provide information about whether roses are included in the subset of flowers that fade quickly. Therefore, based solely on the given information, we cannot definitively conclude that some roses fade quickly. It is possible, but not certain, without additional information.

</details>

---



###  claude-sonnet-4-6
- **Latency:** `5219 ms`
- **Cost:** `$0.003081`

**Preview:** ## Logical Analysis  **No, we cannot conclude that some roses fade quickly.**  This is a classic **logical fallacy** - specifically an **undistributed middle** or invalid syllogism.  ### Why the reaso…

<details>
<summary>Show full response</summary>

## Logical Analysis

**No, we cannot conclude that some roses fade quickly.**

This is a classic **logical fallacy** - specifically an **undistributed middle** or invalid syllogism.

### Why the reasoning fails:

| Premise | Statement |
|---------|-----------|
| P1 | All roses are flowers |
| P2 | Some flowers fade quickly |
| Conclusion? | Some roses fade quickly ❌ |

The "some flowers" that fade quickly **could be entirely different flowers** - tulips, daisies, etc. - with **no overlap with roses whatsoever**.

### Visualizing with sets:

```
Flowers = {roses, tulips, daisies...}
Fades quickly = {tulips, daisies} ← roses need not be here
```

P2 only guarantees *at least one* flower fades

</details>

---



###  claude-haiku-4-5
- **Latency:** `3398 ms`
- **Cost:** `$0.001027`

**Preview:** # No, we cannot conclude that.  Here's why:  **The logical structure:** - All roses are flowers ✓ - Some flowers fade quickly ✓ - Therefore, some roses fade quickly? ✗  **The problem:**  The flowers t…

<details>
<summary>Show full response</summary>

# No, we cannot conclude that.

Here's why:

**The logical structure:**
- All roses are flowers ✓
- Some flowers fade quickly ✓
- Therefore, some roses fade quickly? ✗

**The problem:**

The flowers that fade quickly could be an entirely different group—like tulips or daisies—with no roses among them.

**A concrete example:**
- All roses are flowers ✓
- Some flowers (like daisies) fade quickly ✓
- But all roses might last a long time ✓

This scenario is perfectly consistent with both premises, yet contradicts the conclusion.

**What we'd need:**

To reach the conclusion "some roses fade quickly," we'd need to know that *some of the flowers that fade quickly are specifically roses*—but we don't have that information.

This is a classic logical fallacy sometimes called "illicit minor"

</details>

---



###  claude-opus-4-6
- **Latency:** `6693 ms`
- **Cost:** `$0.005135`

**Preview:** # Analyzing This Syllogism  **No, we cannot validly conclude that some roses fade quickly.**  Here's why:  ## The Logical Structure - **Premise 1:** All roses are flowers (Roses ⊂ Flowers) - **Premise…

<details>
<summary>Show full response</summary>

# Analyzing This Syllogism

**No, we cannot validly conclude that some roses fade quickly.**

Here's why:

## The Logical Structure
- **Premise 1:** All roses are flowers (Roses ⊂ Flowers)
- **Premise 2:** Some flowers fade quickly (Some Flowers ⊂ Things that fade quickly)

## Why the Conclusion Doesn't Follow

The "some flowers" that fade quickly might be **entirely non-rose flowers** — like tulips or daisies. The two premises are compatible with a world where:
- Roses are flowers that *never* fade quickly
- Only lilies and daisies are the flowers that fade quickly

## A Helpful Analogy
- All cats are animals
- Some animals can fly
- ∴ Some cats can fly? — **Clearly not.**

This is a well-

</details>

---


---
## Key Takeaways 📝

| Insight | Detail |
|---------|--------|
| **Speed ≠ Quality** | Faster models may give shorter, simpler answers |
| **Cost varies 10–100×** | gpt-4o-mini costs ~16× less than gpt-4o |
| **Use the cheapest model that works** | Start with mini/haiku, upgrade if quality is insufficient |
| **Multi-provider = resilience** | If one API is down, switch to another |
| **Always benchmark** | Don't assume — measure quality on *your* specific tasks |
| **Alternative providers** | Same SDK works with Groq, Ollama, and others |

---
**Next:** `05_token_explorer.ipynb` — Deep dive into tokenization and cost calculation